In [ ]:
#!/usr/bin/env python3
"""
PE (h^-1) vs Tb (K): 1×2 panels with connected-bin medians
+ 95% bootstrap confidence intervals on the median (shaded envelope)

Left  : PF_rainrate
Right : PF_maxrainrate
Independent y-limits for each panel
"""

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from collections import namedtuple
from scipy.stats import linregress

# ---------- USER PATHS ----------
basedir = '/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/precip_efficiency2/'

# PF_rainrate (left panel)
RR_CWP = basedir + "precip_efficiency_gt1hr_cwp_pf_rainrate_tb.nc"
RR_CLW = basedir + "precip_efficiency_gt1hr_clw_pf_rainrate_tb.nc"
RR_CIW = basedir + "precip_efficiency_gt1hr_ciw_pf_rainrate_tb.nc"

# PF_maxrainrate (right panel)
MX_CWP = basedir + "precip_efficiency_gt1hr_cwp_pf_maxrainrate_tb.nc"
MX_CLW = basedir + "precip_efficiency_gt1hr_clw_pf_maxrainrate_tb.nc"
MX_CIW = basedir + "precip_efficiency_gt1hr_ciw_pf_maxrainrate_tb.nc"

# ---------- SETTINGS ----------
TB_METRIC   = "TB_mean_K"               # choose e.g., "TB_min_K" for deep convection
TB_BINS     = np.arange(210, 231, 2.0)  # K bin edges
MIN_COUNT   = 5                         # min samples per bin
USE_MEDIAN  = True                      # (kept, but bootstrap uses median)
INVERT_X    = True                      # colder Tb on the left

# --- Independent Y-axis limits ---
YLIM_LEFT   = (0, 50)                   # PF_rainrate panel
YLIM_RIGHT  = (0, 200)                  # PF_maxrainrate panel

# --- Regression window ---
TB_MIN = 212.5
TB_MAX = 227.5

# --- Bootstrap settings (median CI) ---
DO_BOOTSTRAP = True
NBOOT = 2000              # 1000–5000 typical
CI_PCT = (2.5, 97.5)      # 95% CI
SEED = 42

# --- Style ---
COLOR_CWP = "#6A5ACD"  # purple
COLOR_CLW = "#FFD60A"  # yellow
COLOR_CIW = "#00A7B7"  # teal
MS        = 80
ALPHA     = 0.95
CI_ALPHA  = 0.22        # shading opacity

Series = namedtuple("Series", "label path color")

# ---------- HELPERS ----------
def flatten(ds, tb_var):
    """Flatten PE and Tb, and mask non-finite values."""
    pe = ds["PRECEFF_TIMEAVG"].values.astype(float).ravel()
    tb = ds[tb_var].values.astype(float).ravel()
    m = np.isfinite(pe) & np.isfinite(tb)
    return pe[m], tb[m]

def binned_stat_bootstrap_median_ci(y, x, bins, min_count=1,
                                   nboot=2000, ci=(2.5, 97.5), seed=42):
    """
    Compute binned median and bootstrap CI for the median.
    Returns mids, median, lo, hi, counts.
    """
    rng = np.random.default_rng(seed)

    mids = 0.5 * (bins[:-1] + bins[1:])
    idx  = np.digitize(x, bins) - 1

    med = np.full(mids.size, np.nan, dtype=float)
    lo  = np.full(mids.size, np.nan, dtype=float)
    hi  = np.full(mids.size, np.nan, dtype=float)
    cnt = np.zeros(mids.size, dtype=int)

    for i in range(mids.size):
        yi = y[idx == i]
        yi = yi[np.isfinite(yi)]
        cnt[i] = yi.size

        if yi.size < min_count:
            continue

        med[i] = np.nanmedian(yi)

        boot = np.empty(nboot, dtype=float)
        for b in range(nboot):
            samp = rng.choice(yi, size=yi.size, replace=True)
            boot[b] = np.nanmedian(samp)

        lo[i] = np.nanpercentile(boot, ci[0])
        hi[i] = np.nanpercentile(boot, ci[1])

    return mids, med, lo, hi, cnt

def binned_stat_simple(y, x, bins, min_count=1, use_median=True):
    """Fallback if you ever want non-bootstrap."""
    mids = 0.5 * (bins[:-1] + bins[1:])
    idx  = np.digitize(x, bins) - 1
    vals = np.full(mids.size, np.nan)
    cnt  = np.zeros(mids.size, dtype=int)
    for i in range(mids.size):
        yi = y[idx == i]
        yi = yi[np.isfinite(yi)]
        cnt[i] = yi.size
        if yi.size >= min_count:
            vals[i] = np.nanmedian(yi) if use_median else np.nanmean(yi)
    return mids, vals, cnt

def plot_series(ax, series, tb_bins, tb_metric, ms, alpha, do_bootstrap=True):
    """
    Plot binned PE vs Tb for one series.
    Returns mids_used, stat_used (for regression), counts_used.
    """
    with xr.open_dataset(series.path) as ds:
        pe, tb = flatten(ds, tb_metric)

    if do_bootstrap:
        mids, med, lo, hi, cnt = binned_stat_bootstrap_median_ci(
            pe, tb, tb_bins,
            min_count=MIN_COUNT,
            nboot=NBOOT,
            ci=CI_PCT,
            seed=SEED
        )
        m = np.isfinite(med)
        mids_u, med_u, lo_u, hi_u, cnt_u = mids[m], med[m], lo[m], hi[m], cnt[m]

        # CI shading
        ax.fill_between(mids_u, lo_u, hi_u, color=series.color, alpha=CI_ALPHA, linewidth=0)

        # Median points/line
        ax.scatter(mids_u, med_u, s=ms, c=series.color, edgecolor="k",
                   linewidths=1.5, alpha=alpha, label=series.label)
        ax.plot(mids_u, med_u, color=series.color, linewidth=1.6, alpha=0.95)

        return mids_u, med_u, cnt_u

    else:
        mids, stat, cnt = binned_stat_simple(pe, tb, tb_bins, MIN_COUNT, USE_MEDIAN)
        m = np.isfinite(stat)
        mids_u, stat_u, cnt_u = mids[m], stat[m], cnt[m]
        ax.scatter(mids_u, stat_u, s=ms, c=series.color, edgecolor="k",
                   linewidths=1.5, alpha=alpha, label=series.label)
        ax.plot(mids_u, stat_u, color=series.color, linewidth=1.6, alpha=0.95)
        return mids_u, stat_u, cnt_u

def add_panel(ax, paths, panel_tag):
    """
    Add all ε series to given axis.
    Returns (stats_for_reg, series_list).
    """
    stats_for_reg = []
    series_list = []

    if paths.get("cwp"):
        series_list.append(Series(r"$\epsilon_{cwp}$", paths["cwp"], COLOR_CWP))
    if paths.get("clw"):
        series_list.append(Series(r"$\epsilon_{\ell}$", paths["clw"], COLOR_CLW))
    if paths.get("ciw"):
        series_list.append(Series(r"$\epsilon_{i}$", paths["ciw"], COLOR_CIW))

    for s in series_list:
        mids, stat, cnt = plot_series(ax, s, TB_BINS, TB_METRIC, MS, ALPHA, do_bootstrap=DO_BOOTSTRAP)
        stats_for_reg.append((mids, stat))

        # Optional diagnostic:
        # print(s.label, "median bin count:", int(np.nanmedian(cnt)), "min:", int(np.nanmin(cnt)), "max:", int(np.nanmax(cnt)))

    if INVERT_X:
        ax.invert_xaxis()

    ax.set_xlabel(r"Brightness temperature $T_b$ [K]", fontsize=20)

    # ticks
    ax.tick_params(axis="x", labelsize=15)
    ax.tick_params(axis="y", labelsize=16)

    # remove spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.2)
    ax.spines["bottom"].set_linewidth(1.2)

    ax.grid(False)

    ax.text(0.02, 0.96, panel_tag, transform=ax.transAxes,
            ha="left", va="top", fontsize=20)

    return stats_for_reg, series_list

def fit_regression_in_range(ax, mids, stat,
                            tb_min=212.5, tb_max=227.5,
                            color="k", label=None):
    """
    Fit a linear regression to points with Tb between tb_min and tb_max,
    plot the regression line, and return (r, p).
    """
    m = (mids >= tb_min) & (mids <= tb_max)
    x = mids[m]
    y = stat[m]

    if x.size < 2:
        print(f"Not enough points for regression in range for {label}.")
        return np.nan, np.nan

    slope, intercept, r_val, p_val, stderr = linregress(x, y)

    x_fit = np.linspace(tb_min, tb_max, 50)
    y_fit = intercept + slope * x_fit
    ax.plot(x_fit, y_fit, color=color, linestyle="--", linewidth=2)

    if label is not None:
        print(f"Regression ({label}) in Tb [{tb_min}, {tb_max}] K:")
    else:
        print(f"Regression in Tb [{tb_min}, {tb_max}] K:")
    print(f"   r = {r_val:.4f},  p = {p_val:.4e}")

    return r_val, p_val

# ---------- PLOT ----------
fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.4), dpi=130)

# --- Left: PF_rainrate ---
stats_rr, series_rr = add_panel(
    axes[0],
    {"cwp": RR_CWP, "clw": RR_CLW, "ciw": RR_CIW},
    r"$\bf{(a)}$ Mean rainrate"
)
axes[0].set_ylabel(r"$\varepsilon$ [h$^{-1}$]", fontsize=22)
axes[0].set_ylim(*YLIM_LEFT)

# --- Right: PF_maxrainrate ---
stats_mx, series_mx = add_panel(
    axes[1],
    {"cwp": MX_CWP, "clw": MX_CLW, "ciw": MX_CIW},
    r"$\bf{(b)}$ Max-rainrate"
)
axes[1].set_ylim(*YLIM_RIGHT)

# --- Fit linear regression for each ε metric between TB_MIN and TB_MAX ---
print("\n=== PF_rainrate panel ===")
for (mids, stat), s in zip(stats_rr, series_rr):
    fit_regression_in_range(
        axes[0], mids, stat,
        tb_min=TB_MIN, tb_max=TB_MAX,
        color=s.color, label=f"Rainrate {s.label}"
    )

print("\n=== PF_maxrainrate panel ===")
for (mids, stat), s in zip(stats_mx, series_mx):
    fit_regression_in_range(
        axes[1], mids, stat,
        tb_min=TB_MIN, tb_max=TB_MAX,
        color=s.color, label=f"Max-rainrate {s.label}"
    )

# --- Legend (on right panel only) ---
handles, labels = axes[1].get_legend_handles_labels()
axes[1].legend(handles, labels, frameon=False, loc="center left", fontsize=15)

# --- rotate x tick labels ---
for ax in axes:
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right")

plt.tight_layout(w_pad=1.2)

# --- Save ---
fig.savefig("/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_Paper_Plots/Figure8_PE_cloud_top_temp.png",
             dpi=200, bbox_inches="tight")
fig.savefig("/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_Paper_Plots/Figure8_PE_cloud_top_temp.pdf",
             dpi=200, bbox_inches="tight")
